In [ ]:
import pandas as pd
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision import transforms
import numpy as np
import bvtrain as bv
import os
import time
import matplotlib.pyplot as plt

os.makedirs("checkpoints", exist_ok=True)
env = bv.setup()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# speed tip 1: use all available CPU threads for torch ops
torch.set_num_threads(os.cpu_count())
print(f"Using {os.cpu_count()} CPU threads")

MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 16          # speed tip 2: smaller batch = faster per-step on CPU (less memory pressure)
EPOCHS = 3               # speed tip 3: fewer epochs — cosine schedule still anneals smoothly
LR = 3e-5
N_SPECIES = 100          # speed tip 4: subset for fast iteration — set to None only for your final full run

data = bv.load_data(env, n_species=N_SPECIES)
hf = data._hf
labels = data.labels
num_labels = data.n_labels
print(f"{num_labels} species | train={len(hf['train'])} val={len(hf['val'])} test={len(hf['test'])}")

# preprocessing
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

class TransformFn:
    def __init__(self, tfm):
        self.tfm = tfm
    def __call__(self, batch):
        batch["pixel_values"] = [self.tfm(img.convert("RGB")) for img in batch["image"]]
        return batch

train_ds = hf["train"].with_transform(TransformFn(train_transform))
val_ds   = hf["val"].with_transform(TransformFn(eval_transform))
test_ds  = hf["test"].with_transform(TransformFn(eval_transform))

def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    labels_t = torch.tensor([b["label"] for b in batch])
    return {"pixel_values": pixel_values, "labels": labels_t}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

# model
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)
model.to(device)

# speed tip 5: freeze most of the pretrained backbone, only fine-tune the last few blocks + classifier
# this drastically cuts backward-pass compute since most of the network no longer needs gradients
FREEZE_UNTIL_LAYER = 9   # freeze blocks 0-8 (of 0-11), only train blocks 9-11 + classifier head
for name, param in model.named_parameters():
    if "classifier" in name:
        continue  # always train the head
    # match either naming scheme (encoder.layer.N or vit.layers.N) depending on your transformers version
    is_late_block = any(f".{i}." in name for i in range(FREEZE_UNTIL_LAYER, 12))
    param.requires_grad = is_late_block

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Training {trainable:,} / {total:,} parameters ({100*trainable/total:.1f}%)")

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],  # only pass trainable params
    lr=LR, weight_decay=0.01
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# training/eval
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_t = batch["labels"].to(device)
            outputs = model(pixel_values=pixel_values).logits
            loss = criterion(outputs, labels_t)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * pixel_values.size(0)
            correct += (outputs.argmax(-1) == labels_t).sum().item()
            total += pixel_values.size(0)
    return total_loss / total, correct / total

best_val_acc = 0.0
for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    elapsed = time.time() - t0
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.3f} | val loss {val_loss:.3f} acc {val_acc:.3f} | {elapsed:.1f}s")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "checkpoints/vit_species_best.pt")
        print("  -> saved new best checkpoint")

def evaluate_topk(loader, k=5):
    model.eval()
    top1, top5, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device)
            labels_t = batch["labels"].to(device)
            outputs = model(pixel_values=pixel_values).logits
            _, topk_preds = outputs.topk(k, dim=-1)
            top1 += (topk_preds[:, 0] == labels_t).sum().item()
            top5 += (topk_preds == labels_t.unsqueeze(1)).any(dim=1).sum().item()
            total += labels_t.size(0)
    return top1 / total, top5 / total

model.load_state_dict(torch.load("checkpoints/vit_species_best.pt"))
top1, top5 = evaluate_topk(test_loader, k=5)
print(f"Test top-1: {top1:.3f} | top-5: {top5:.3f}")

In [ ]:
model.eval()

N_EXAMPLES = 6
examples = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels_t = batch["labels"].to(device)
        outputs = model(pixel_values=pixel_values).logits
        preds = outputs.argmax(-1)
        top5_preds = outputs.topk(5, dim=-1).indices

        for i in range(pixel_values.size(0)):
            if len(examples) >= N_EXAMPLES:
                break
            examples.append({
                "pixel_values": pixel_values[i].cpu(),
                "true_label": labels_t[i].item(),
                "pred_label": preds[i].item(),
                "top5": top5_preds[i].cpu().tolist(),
            })
        if len(examples) >= N_EXAMPLES:
            break

# undo normalization so the image displays with correct colors
mean = torch.tensor(processor.image_mean).view(3, 1, 1)
std = torch.tensor(processor.image_std).view(3, 1, 1)

def denormalize(img_tensor):
    img = img_tensor * std + mean
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, ex in enumerate(examples):
    img = denormalize(ex["pixel_values"])
    true_species = labels[ex["true_label"]]
    pred_species = labels[ex["pred_label"]]
    correct = ex["true_label"] == ex["pred_label"]
    in_top5 = ex["true_label"] in ex["top5"]

    axes[idx].imshow(img)
    color = "green" if correct else ("orange" if in_top5 else "red")
    title = f"True: {true_species}\nPred: {pred_species}"
    axes[idx].set_title(title, fontsize=9, color=color)
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig("checkpoints/vit_example_predictions.png", dpi=150, bbox_inches="tight")
plt.show()